In [2]:
import os
from langchain_community.document_loaders import UnstructuredFileLoader
from loguru import logger

# Cấu hình đường dẫn
INPUT_DIR = "./data_input"  # Nơi chứa file DOCX, PDF, TXT, MD
processed_docs = []

def ingest_documents(directory):
    if not os.path.exists(directory):
        logger.error(f"Thư mục {directory} không tồn tại!")
        return []

    for filename in os.listdir(directory):
        file_path = os.path.join(directory, filename)
        
        # Chỉ xử lý các định dạng trong phạm vi dự án
        if filename.endswith(('.pdf', '.docx', '.txt', '.md')):
            try:
                logger.info(f"Đang xử lý file: {filename}")
                
                # Sử dụng Unstructured để tự động nhận diện và parse
                loader = UnstructuredFileLoader(
                    file_path, 
                    mode="single",  # Gộp toàn bộ nội dung file thành 1 object Document
                    strategy="fast" # Có thể đổi thành "hi_res" nếu file PDF có bảng biểu phức tạp
                )
                
                docs = loader.load()
                
                # Bổ sung/Chuẩn hóa Metadata cho Người 2
                for doc in docs:
                    doc.metadata["source_file"] = filename
                    doc.metadata["file_type"] = filename.split('.')[-1]
                    processed_docs.append(doc)
                
                logger.success(f"Hoàn thành ingest: {filename}")
                
            except Exception as e:
                logger.error(f"Lỗi khi xử lý {filename}: {str(e)}")
        else:
            logger.warning(f"Bỏ qua định dạng không hỗ trợ: {filename}")

    return processed_docs

# Chạy thử
if __name__ == "__main__":
    all_data = ingest_documents(INPUT_DIR)
    print(f"\nTổng số tài liệu đã xử lý: {len(all_data)}")
    if all_data:
        print("Ví dụ metadata của file đầu tiên:", all_data[0].metadata)

2026-03-18 23:43:06.721 | INFO     | __main__:ingest_documents:20 - Đang xử lý file: docs_phan_chia_cong_viec_chatbot_rag.docx
C:\Users\quant\AppData\Local\Temp\ipykernel_2896\904825401.py:23: LangChainDeprecationWarning: The class `UnstructuredFileLoader` was deprecated in LangChain 0.2.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-unstructured package and should be used instead. To use it run `pip install -U `langchain-unstructured` and import as `from `langchain_unstructured import UnstructuredLoader``.
  loader = UnstructuredFileLoader(
2026-03-18 23:43:14.964 | SUCCESS  | __main__:ingest_documents:37 - Hoàn thành ingest: docs_phan_chia_cong_viec_chatbot_rag.docx
2026-03-18 23:43:14.966 | INFO     | __main__:ingest_documents:20 - Đang xử lý file: English for Career Development Syllabus 2023.pdf


2026-03-18 23:43:22.618 | SUCCESS  | __main__:ingest_documents:37 - Hoàn thành ingest: English for Career Development Syllabus 2023.pdf
2026-03-18 23:43:22.619 | INFO     | __main__:ingest_documents:20 - Đang xử lý file: Part1-4_Minh_EN.md
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
2026-03-18 23:43:23.940 | SUCCESS  | __main__:ingest_documents:37 - Hoàn thành ingest: Part1-4_Minh_EN.md
2026-03-18 23:43:23.941 | INFO     | __main__:ingest_documents:20 - Đang xử lý file: Part1-4_Minh_VN.md
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
2026-03-18 23:43:24.667 | SUCCESS  | __main__:ingest_documents:37 - Hoàn thành ingest: Part1-4_Minh_VN.md
2026-03-18 23:43:24.668 | WARNING  | __main__:ingest_documents:42 - Bỏ qua định dạng không hỗ trợ: SOLUTIONS.Chapters1-5.doc
2026-03-18 23:43:24.670 | INFO     | __main__:ingest_documents:20 - Đang xử lý file: test


Tổng số tài liệu đã xử lý: 5
Ví dụ metadata của file đầu tiên: {'source': './data_input\\docs_phan_chia_cong_viec_chatbot_rag.docx', 'source_file': 'docs_phan_chia_cong_viec_chatbot_rag.docx', 'file_type': 'docx'}


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings # Yêu cầu: pip install sentence-transformers

# 1. SPLIT: Chia nhỏ tài liệu thành các đoạn (chunks)
# Giúp AI dễ dàng tìm kiếm và không bị quá giới hạn token của LLM
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,    # Độ dài mỗi chunk (khoảng 1000 ký tự)
    chunk_overlap=100,  # Đoạn gối đầu giữa các chunk để không mất ngữ cảnh
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(all_data)

# 1. Setup
db_directory = "./vector_db"
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# OPTION A: If you want to start FRESH every time (Safest for learning)
if os.path.exists(db_directory):
    print("Removing old database to start fresh...")
    shutil.rmtree(db_directory)
# Now create the DB
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=db_directory
)
# OPTION B: If you want to LOAD an existing DB instead of re-creating
# (Use this once your dataset gets very large)
"""
if os.path.exists(db_directory):
    vector_db = Chroma(persist_directory=db_directory, embedding_function=embeddings)
    print("Existing database loaded.")
else:
    vector_db = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_directory)
    print("New database created.")
"""


print(f"Thành công! Đã chia {len(all_data)} tài liệu thành {len(chunks)} chunks.")
print(f"Vector Database đã được lưu tại: {db_directory}")

# 3. THỬ NGHIỆM TRUY XUẤT (Similarity Search)
query = "Mục tiêu của khóa học English for Career Development là gì?"
results = vector_db.similarity_search(query, k=2)

print("\n--- Kết quả tìm kiếm thử nghiệm ---")
for i, doc in enumerate(results):
    print(f"\nKết quả {i+1} (Nguồn: {doc.metadata.get('source_file')}):")
    print(f"Nội dung: {doc.page_content[:200]}...")



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Thành công! Đã chia 5 tài liệu thành 41 chunks.
Vector Database đã được lưu tại: ./vector_db

--- Kết quả tìm kiếm thử nghiệm ---

Kết quả 1 (Nguồn: docs_phan_chia_cong_viec_chatbot_rag.docx):
Nội dung: 9. Kết luận

Phương án phân chia theo 5 lane phù hợp với nhóm 5 người triển khai chatbot RAG theo pipeline kỹ thuật rõ ràng. Cách chia này giúp xác lập ownership cụ thể, giảm chồng lấn trách nhiệm, hỗ...

Kết quả 2 (Nguồn: English for Career Development Syllabus 2023.pdf):
Nội dung: By the end of this course, you will be able to:

• • • • •

Compare and contrast the job search process in the United States and your home country. Describe your professional interests and talents and...
